# CountRecall · accuracy inside each summary boundary segment

What a query costs as it moves away from the boundary that rewrote the summary. The CountRecall Medium stream is 103 scored queries under the fixed lazy C32 schedule, so it falls into four segments (queries 1–32, 33–64, 65–96, 97–103) and the writes precede queries 33, 65 and 97. Each panel is one segment: exact accuracy against the query's position inside it, on the frozen 8M endpoints of the confirmation roster, so segment 1 reads what a cell can do with the evidence still in its buffer and segments 2–4 read what survived one, two and three rewrites. The method's summary-cleared panel is drawn beside it as the dotted trace: the same policy with the carried summary reset at every boundary, the floor the carry has to beat.

The figure draws the **summary carriers and their controls**: RSM-O, RSM no carry, the GRU, Memo and Memo with fixed segments. The full-history Transformer is left out and the figure says so under the panels — it keeps every record, so its accuracy does not depend on where a boundary falls; it and the rewrite ablation (RSM-R) are drawn in the appendix figures. Curves carry no markers, as in the archived boundary-segment display; only the summary-cleared trace keeps its crosses, since it runs along zero where a bare line would be lost against the axis.

Every cell is scored on the same queries and the same roster. Uncertainty is the study's joint seed/task bootstrap of the per-stream values (95 %, conditional on the three trained seeds); whole streams are resampled, never the queries inside a stream. A cell without records on the chosen roster is named in the legend as pending and is never filled from another roster; nothing about it is written on a chart. The estimator and the plotting code are in this notebook, the shared plumbing (record paths, the roster rule, the palette, the save step) in [figures_common.py](figures_common.py); rules are stated in the notebook.

The companion retention view — accuracy by the number of C32 rewrites since the queried suit was *first dealt*, which sorts the same queries by the age of their evidence instead of by their position — is the CountRecall column of [figure3_beyond_the_training_horizon](figure3_beyond_the_training_horizon.ipynb) and its appendix figure.

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
from collections import defaultdict
from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from figures_common import (
    ABLATION_CONDITION,
    DRAWN_CELLS,
    ENVIRONMENTS,
    FULL_WIDTH,
    GRID,
    INK_MUTED,
    METHOD_CONDITION,
    PANEL,
    SEEDS,
    SEGMENT,
    SUMMARY_CLEARED,
    BuiltFigure,
    CellStyle,
    band,
    choose_split,
    composite_legend,
    line_handles,
    load_cell,
    paper_style,
    pending_handles,
    standalone,
    style_axes,
)

from reasoned_icrl.analysis.statistics import _exact_estimate

SEGMENT_CONDITIONS: tuple[str, ...] = (
    METHOD_CONDITION,
    "raw_segment",
    "full_gru",
    "memo",
    "memo_fixed",
)
"""The cells this figure draws: RSM-O, its carry
control and the recurrent and summary comparators. The full-history
Transformer is left out — the comparison here is against the *summary*
carriers, Memo and Memo fixed, and the full history keeps every record, so it
has no boundary to be measured against."""
SEGMENT_CELLS: tuple[CellStyle, ...] = tuple(
    c for c in DRAWN_CELLS if c.condition in SEGMENT_CONDITIONS
)
OMITTED_NOTE = (
    "The full-history Transformer is not drawn here: it retains every record, "
    "so its accuracy does not depend on where a boundary falls.\nIt and the "
    "rewrite ablation (RSM-R) are in the appendix figures."
)
"""Printed under the figure, so what is missing is said and not implied."""


@dataclass(frozen=True)
class BoundarySegmentOptions:
    seeds: Sequence[int] = SEEDS
    samples: int = 2000
    resampling_seed: int = 0
    split_preference: Sequence[str] = ("confirmation", "development")
    cells: Sequence[CellStyle] = SEGMENT_CELLS
    """RSM-O, its carry control, the GRU and the two Memo cells
    (:data:`SEGMENT_CONDITIONS`)."""
    method: str = METHOD_CONDITION
    """The paper's RSM (the residual rewrite): the cell whose summary-cleared
    panel is drawn as the floor."""
    segment_length: int = SEGMENT
    """Records per segment (C = 32): the lazy boundary opens at record 33."""
    caption: str = OMITTED_NOTE
    """Printed under the composite; empty prints nothing."""
    name: str = "figure_countrecall_boundary_segments"
    figsize: tuple[float, float] = (FULL_WIDTH, 3.5)
    panel_size: tuple[float, float] = PANEL


# ---------------------------------------------------------------- estimator
# A query's segment and its position inside that segment come from the scored
# query index alone: the schedule is lazy and fixed (writes precede queries 33,
# 65, 97), so every cell and every stream share it and no cell's own write
# record is needed. Accuracy is a mean over the (seed, task, rollout) streams,
# one value per stream per position; the interval is the study's joint
# seed/task bootstrap, conditional on the trained seeds.


def segment_of(event_index, *, segment_length):
    """(segment, position) of a scored query: segment 1 is the queries whose
    evidence is still in the buffer, segment s > 1 the queries asked after
    s - 1 rewrites."""
    index = int(event_index) - 1
    return index // segment_length + 1, index % segment_length + 1


def boundary_segment_rows(events, *, segment_length, samples, seed, condition=None):
    """Per condition, segment and within-segment position: exact accuracy over
    the streams, its 95 % interval, the per-seed values and the streams behind
    it."""
    rng = np.random.default_rng(seed)
    rows = []
    for name in sorted(
        {e.condition for e in events} if condition is None else {condition}
    ):
        mine = [e for e in events if e.kind == "query" and e.condition == name]
        grouped = defaultdict(dict)
        for e in mine:
            segment, position = segment_of(e.event_index, segment_length=segment_length)
            cell = (e.training_seed, e.task_id, e.rollout_seed)
            grouped[(segment, position)][cell] = float(e.numerator)
        for (segment, position), values in sorted(grouped.items()):
            estimate, lower, upper, streams, seeds, per_seed = _exact_estimate(
                values, samples=samples, confidence=0.95, rng=rng
            )
            rows.append(
                {
                    "condition": name,
                    "segment": segment,
                    "position": position,
                    "query": (segment - 1) * segment_length + position,
                    "estimate": estimate,
                    "lower": lower,
                    "upper": upper,
                    "streams": streams,
                    "seeds": len(seeds),
                    "per_seed": {str(s): round(v, 6) for s, v in per_seed.items()},
                }
            )
    return rows


def segment_spans(rows, *, segment_length):
    """The query range each drawn segment covers, for its panel title."""
    spans = {}
    for row in rows:
        first, last = spans.get(row["segment"], (row["query"], row["query"]))
        spans[row["segment"]] = (min(first, row["query"]), max(last, row["query"]))
    return dict(sorted(spans.items()))


# ------------------------------------------------------------------ compute


def compute_boundary_segments(layout, options, sources):
    """The retained rows of every present cell and the method's
    summary-cleared rows, on one roster."""
    contract = layout.contract("countrecall")
    selection = choose_split(
        layout,
        "countrecall",
        [c.condition for c in options.cells],
        contract=contract,
        preference=options.split_preference,
        seeds=options.seeds,
        sources=sources,
    )
    present = [c for c in options.cells if c.condition in selection.present]
    data = {
        "selection": selection,
        "present": present,
        "missing": [c for c in options.cells if c.condition in selection.missing],
        "rows": [],
        "cleared": [],
        "spans": {},
        "segment_length": options.segment_length,
    }
    if not present:
        return data
    events = [e for c in present for e in selection.panels[c.condition]]
    data["rows"] = boundary_segment_rows(
        events,
        segment_length=options.segment_length,
        samples=options.samples,
        seed=options.resampling_seed,
    )
    data["spans"] = segment_spans(data["rows"], segment_length=options.segment_length)
    cleared = load_cell(
        layout,
        "countrecall",
        options.method,
        selection.split,
        "summary-cleared",
        contract=contract,
        seeds=options.seeds,
        sources=sources,
    )
    if cleared is not None:
        data["cleared"] = boundary_segment_rows(
            cleared,
            segment_length=options.segment_length,
            samples=options.samples,
            seed=options.resampling_seed,
            condition=options.method,
        )
    return data


# ------------------------------------------------------------------- drawing


def draw_segment(ax, data, options, segment, *, first_column=True, xlabel=True):
    """One segment's panel: accuracy against the within-segment query position,
    a band per cell, the method's summary-cleared trace dotted. In the
    composite the columns share one y axis and one x label; a standalone panel
    carries both."""
    spans = data["spans"]
    for cell in data["present"]:
        mine = sorted(
            (
                r
                for r in data["rows"]
                if r["condition"] == cell.condition and r["segment"] == segment
            ),
            key=lambda r: r["position"],
        )
        if not mine:
            continue
        x = [r["position"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.line())
    cleared = sorted(
        (r for r in data["cleared"] if r["segment"] == segment),
        key=lambda r: r["position"],
    )
    if cleared:
        # The cleared trace runs along zero, where a bare line would be lost
        # against the axis: it keeps the crosses that name it in the legend.
        ax.plot(
            [r["position"] for r in cleared],
            [r["estimate"] for r in cleared],
            **SUMMARY_CLEARED.markers(3.8, markevery=6),
        )
    span = spans.get(segment)
    width = max(
        (r["position"] for r in data["rows"] if r["segment"] == segment), default=1
    )
    ax.set_xlim(0.5, width + 0.5)
    ticks = [1, *range(8, width + 1, 8)] if width > 8 else list(range(1, width + 1))
    ax.set_xticks(ticks)
    ax.set_ylim(0, 1.03)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    if first_column:
        ax.set_ylabel("Exact accuracy")
    if xlabel:
        ax.set_xlabel("Query position in the segment")
    rewrites = segment - 1
    kept = (
        "evidence in the buffer"
        if rewrites == 0
        else f"{rewrites} rewrite" + ("s" if rewrites > 1 else "") + " ago"
    )
    title = f"Segment {segment}"
    if span:
        title += f" · queries {span[0]}-{span[1]}"
    ax.set_title(f"{title}\n{kept}", fontsize=9.5)
    style_axes(ax)


def draw_all_segments(ax, data, options):
    """Every segment on one axis, against the scored query index: the
    standalone form of the same numbers, with the write boundaries marked."""
    for cell in data["present"]:
        mine = sorted(
            (r for r in data["rows"] if r["condition"] == cell.condition),
            key=lambda r: r["query"],
        )
        if not mine:
            continue
        x = [r["query"] for r in mine]
        band(ax, x, [r["lower"] for r in mine], [r["upper"] for r in mine], cell)
        ax.plot(x, [r["estimate"] for r in mine], **cell.line())
    if data["cleared"]:
        rows = sorted(data["cleared"], key=lambda r: r["query"])
        ax.plot(
            [r["query"] for r in rows],
            [r["estimate"] for r in rows],
            **SUMMARY_CLEARED.markers(3.8, markevery=12),
        )
    length = options.segment_length
    last = max((r["query"] for r in data["rows"]), default=length)
    for boundary in range(length, last, length):
        ax.axvline(boundary + 0.5, color=GRID, linewidth=0.5, zorder=0)
    ax.set_xlim(0.5, last + 0.5)
    ax.set_xticks([1, *range(length, last + 1, length)])
    ax.set_ylim(0, 1.03)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("Scored query index (boundaries marked)")
    ax.set_ylabel("Exact accuracy")
    ax.set_title(data["selection"].title(ENVIRONMENTS["countrecall"].title))
    style_axes(ax)


# --------------------------------------------------------------------- build


def build_boundary_segments(layout, options=None):
    """The boundary-segment figure from the saved records under ``layout``:
    one panel per C32 segment, every cell on the same queries."""
    options = options or BoundarySegmentOptions()
    sources = []
    data = compute_boundary_segments(layout, options, sources)
    segments = list(data["spans"]) or [1]
    # The legend shows what is drawn: bare lines, since the curves carry no
    # markers; the cleared trace keeps the crosses it is drawn with.
    handles = line_handles(data["present"]) + (
        [SUMMARY_CLEARED.handle()] if data["cleared"] else []
    )
    handles += pending_handles(
        {ENVIRONMENTS["countrecall"].title: data["missing"]},
        options.cells,
        reason="pending (no records on the roster)",
    )
    with paper_style():
        fig = plt.figure(figsize=options.figsize, layout="constrained")
        axes = np.atleast_1d(fig.subplots(1, len(segments), sharey=True))
        for column, segment in enumerate(segments):
            draw_segment(
                axes[column],
                data,
                options,
                segment,
                first_column=column == 0,
                xlabel=False,
            )
        fig.supxlabel("Query position in the segment", fontsize=10)
        composite_legend(fig, handles, ncol=4)
        if options.caption:
            fig.text(
                0.5,
                -0.01,
                options.caption,
                ha="center",
                va="top",
                fontsize=8.5,
                color=INK_MUTED,
            )
    panels = {
        "by_query": standalone(
            lambda ax: draw_all_segments(ax, data, options),
            size=options.panel_size,
            handles=handles,
        ),
    }
    for segment in segments:
        panels[f"segment_{segment}"] = standalone(
            lambda ax, s=segment: draw_segment(ax, data, options, s, first_column=True),
            size=options.panel_size,
            handles=handles,
        )
    selection = data["selection"]
    tables = {
        "accuracy": [
            {**r, "history": "retained", "split": selection.split} for r in data["rows"]
        ]
        + [
            {**r, "history": "summary-cleared", "split": selection.split}
            for r in data["cleared"]
        ]
    }
    notes = {
        "countrecall": selection.notes(),
        "segments": {str(k): list(v) for k, v in data["spans"].items()},
        "segment_length": options.segment_length,
        "pending_in_legend": [c.condition for c in data["missing"]],
        "cells": [c.condition for c in options.cells],
        "omitted": {
            "full_context": "retains every record; no boundary to measure",
            ABLATION_CONDITION: "the rewrite ablation (RSM-R), drawn in the appendix",
        },
        "seeds": list(options.seeds),
        "samples": options.samples,
        "resampling_seed": options.resampling_seed,
        "method": options.method,
        "checkpoint": "policy_epoch_999 (collection endpoint)",
        "interval": "95 % joint seed/task bootstrap, conditional on the trained seeds",
        "unit": "one value per (seed, task, rollout) stream per position",
    }
    return BuiltFigure(options.name, fig, tables, notes, sources, panels)

In [ ]:
# Parameters: `cells` restricts the drawn cells (fixed order; the default is
# RSM-O, its carry control, the GRU and the two Memo cells — the summary
# carriers this figure compares), `method` the cell whose summary-cleared panel
# is the floor, `caption` the note printed under the figure, `samples` the
# bootstrap draw count (2000 in the paper).
options = BoundarySegmentOptions(samples=2000, resampling_seed=0, cells=SEGMENT_CELLS)
boundary_segments = build_boundary_segments(layout, options)
display(boundary_segments.figure)

In [ ]:
# The roster the panel sits on, the segments read and the cells pending there.
print(boundary_segments.notes["countrecall"])
print("segments:", boundary_segments.notes["segments"])

In [ ]:
# The plotted numbers: accuracy at the first and last position of each segment.
rows = boundary_segments.tables["accuracy"]
edges = [
    r
    for r in rows
    if r["history"] == "retained"
    and r["position"]
    in (1, max(x["position"] for x in rows if x["segment"] == r["segment"]))
]
display(
    Markdown(
        markdown_preview(
            sorted(edges, key=lambda r: (r["segment"], r["position"], r["condition"])),
            limit=24,
        )
    )
)

In [ ]:
# Save: the composite (PDF + PNG), every panel (PNG), the accuracy table and
# the notes.
for path in boundary_segments.save(FIGURES_DIR, DATA_DIR):
    print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
boundary_segments.close()